# Capitolo 5 (parte 1) — CNN da zero su CIFAR-10

In [ ]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import numpy as np
import matplotlib.pyplot as plt

import torch, time
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
fissa_seme(42)

## 5.2 CIFAR-10 — download ufficiale

In [ ]:
norm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
train_ds = datasets.CIFAR10("../data", train=True,  download=True, transform=norm)
test_ds  = datasets.CIFAR10("../data", train=False, download=True, transform=norm)
classi = train_ds.classes
print(classi)

**Alternativa** se il download dal server ufficiale è troppo lento: stesso dataset dal mirror Hugging Face (formato Parquet). Eseguire questa cella al posto della precedente.

In [ ]:
# import sys; sys.path.insert(0, "..")
# import dati
# def prepara(X, y):
#     X = torch.from_numpy(X).permute(0, 3, 1, 2).float() / 255; return TensorDataset((X - 0.5) / 0.5, torch.from_numpy(y.copy()).long())
# Xa, ya, Xb, yb = dati.cifar10_mirror(); train_ds, test_ds = prepara(Xa, ya), prepara(Xb, yb)
# classi = dati.CIFAR10_CLASSI

In [ ]:
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)   # su Windows: num_workers=0
test_dl  = DataLoader(test_ds,  batch_size=500, num_workers=2)
img, et = train_ds[0]
print(img.shape, classi[et])
fig, assi = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(assi.flat):
    im, e = train_ds[i]
    ax.imshow((im.permute(1, 2, 0) * 0.5 + 0.5).numpy()); ax.set_title(classi[e], fontsize=8); ax.axis("off")
plt.show()

## 5.3 Il modello

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),  # → 32×16×16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # → 64×8×8
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # → 128×4×4
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.fc(self.conv(x))

x = torch.zeros(1, 3, 32, 32)
for strato in CNN().conv:
    x = strato(x)
    if not isinstance(strato, nn.ReLU):
        print(f"{type(strato).__name__:10s}", tuple(x.shape))

## 5.4 Il confronto MLP / CNN

In [ ]:
perdita_fn = nn.CrossEntropyLoss()

def valuta(m):
    m.eval(); corretti, perdita = 0, 0.0
    with torch.no_grad():
        for xb, yb in test_dl:
            out = m(xb); perdita += perdita_fn(out, yb).item() * len(yb)
            corretti += (out.argmax(1) == yb).sum().item()
    return perdita / len(test_ds), corretti / len(test_ds)

def addestra(m, epoche, lr=1e-3):
    opt = torch.optim.Adam(m.parameters(), lr=lr); t0 = time.time(); storia = []
    for e in range(epoche):
        m.train(); somma, n = 0.0, 0
        for xb, yb in train_dl:
            perdita = perdita_fn(m(xb), yb)
            opt.zero_grad(); perdita.backward(); opt.step()
            somma += perdita.item() * len(yb); n += len(yb)
        pt, at = valuta(m); storia.append((somma / n, pt, at))
        print(f"Epoca {e+1}: train {somma/n:.4f} | test {pt:.4f} | acc {at:.2%} ({time.time()-t0:.0f}s)")
    return storia

In [ ]:
fissa_seme(42)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(3072, 512), nn.ReLU(), nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 10))
print("Parametri MLP:", sum(p.numel() for p in mlp.parameters()))
storia_mlp = addestra(mlp, 5)

In [ ]:
fissa_seme(42)
cnn = CNN()
print("Parametri CNN:", sum(p.numel() for p in cnn.parameters()))
storia_cnn = addestra(cnn, 6)
torch.save(cnn.state_dict(), "cnn_cifar10.pt")

In [ ]:
plt.plot([s[2] for s in storia_mlp], marker="o", label="MLP"); plt.plot([s[2] for s in storia_cnn], marker="o", label="CNN")
plt.xlabel("Epoca"); plt.ylabel("Accuratezza test"); plt.legend(); plt.grid(True); plt.show()

### Accuratezza per classe

In [ ]:
cnn.eval(); corretti = np.zeros(10); totali = np.zeros(10)
with torch.no_grad():
    for xb, yb in test_dl:
        pred = cnn(xb).argmax(1)
        for a, b in zip(yb.tolist(), pred.tolist()):
            totali[a] += 1; corretti[a] += (a == b)
for c, v in zip(classi, corretti / totali):
    print(f"{c:12s} {v:.1%}")

## 5.7 Esperimento: data augmentation + batch normalization (lento: ~3 min/epoca su 4 core)

In [ ]:
# aug = transforms.Compose([
#     transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
#     transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
# ])
# train_dl = DataLoader(datasets.CIFAR10("../data", train=True, transform=aug), batch_size=64, shuffle=True, num_workers=2)
#
# class CNNbn(nn.Module):
#     def __init__(self):
#         super().__init__()
#         def blocco(i, o):
#             return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
#         self.conv = nn.Sequential(blocco(3, 32), blocco(32, 64), blocco(64, 128))
#         self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(2048, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 10))
#     def forward(self, x):
#         return self.fc(self.conv(x))
#
# fissa_seme(42)
# storia_bn = addestra(CNNbn(), 30)